# Phase 2 - Full Fine-Tuning

Phase 1 trained only the new output layer while the MobileNetV2 backbone was locked. Now we load those weights, unlock everything, and train the whole network together at a much lower learning rate.

The lower learning rate matters here. Phase 1 used 1e-3. If we kept that rate now, we would destroy the useful patterns the backbone learned from ImageNet. At 1e-5 the backbone adjusts slowly to parking-specific patterns instead of forgetting what it knows.

**Before running this notebook on Kaggle:**
- Attach the dataset `https://www.kaggle.com/datasets/raahad/parking-occupancy-merged` as input
- Attach the dataset `https://www.kaggle.com/datasets/raahad/mobilenetv2-parking-occupancy-phase1-weights` as input

## Imports

In [1]:
import os
import gc
import csv
import json
import sys
import subprocess
import torch
import torch.nn as nn
import torch.utils.checkpoint as ckpt
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

print("PyTorch version:", torch.__version__)
print("GPU available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

import psutil
def log_mem(msg=""):
    gb = 1024**3
    ram = psutil.virtual_memory()
    print(f"[MEM {msg}] RAM {ram.used/gb:.1f}/{ram.total/gb:.1f} GB ({ram.percent}%)")
    if torch.cuda.is_available():
        r = torch.cuda.memory_reserved(0) / gb
        a = torch.cuda.memory_allocated(0) / gb
        print(f"  GPU: allocated {a:.2f} GB, reserved {r:.2f} GB")
    sys.stdout.flush()

PyTorch version: 2.10.0+cu128
GPU available:   True
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## Settings

In [ ]:
DATA_DIR  = "/kaggle/input/datasets/raahad/parking-occupancy-merged"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR   = os.path.join(DATA_DIR, "val")

CHECKPOINT_PATH  = "/kaggle/input/datasets/raahad/mobilenetv2-parking-occupancy-phase1-weights/phase1_weights.pth"

KAGGLE_USERNAME  = "your-username"
DATASET_NAME     = "mobilenetv2-parking-occupancy-phase2-weights"
EXPORT_DIR       = f"/kaggle/working/{DATASET_NAME}"
WEIGHTS_FILENAME = "phase2_weights.pth"

EPOCHS        = 10
BATCH_SIZE    = 128   # same as phase1
LEARNING_RATE = 1e-5  # low to protect pretrained backbone features
NUM_WORKERS   = 2     # 2 workers

USE_AMP = torch.cuda.is_available()  # mixed precision only on GPU

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", DEVICE)
print("AMP enabled:", USE_AMP)

Running on: cuda
AMP enabled: True


## Load the Dataset

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=val_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=USE_AMP,
    persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=USE_AMP,
    persistent_workers=(NUM_WORKERS > 0)
)

print("Classes:", train_dataset.classes)
print("Train samples:", len(train_dataset))
print("Val samples:  ", len(val_dataset))

Classes: ['empty', 'occupied']
Train samples: 624559
Val samples:   184432


## Load Phase 1 Weights, Unlock All Layers, Enable Gradient Checkpointing

Gradient checkpointing splits the backbone into 2 segments and recomputes
activations during backward instead of storing them.

In [ ]:
model = models.mobilenet_v2(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(model.last_channel, 2)
)

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
print(f"Loaded: {CHECKPOINT_PATH}")

# Unlock everything
for param in model.parameters():
    param.requires_grad = True

# Gradient checkpointing — 2-segment split over backbone layers
# RAM Optimizer
_layers = list(model.features.children())
_mid    = len(_layers) // 2
_seg1   = nn.Sequential(*_layers[:_mid])
_seg2   = nn.Sequential(*_layers[_mid:])

class CheckpointedFeatures(nn.Module):
    def __init__(self, seg1, seg2):
        super().__init__()
        self.seg1 = seg1
        self.seg2 = seg2
    def forward(self, x):
        x = ckpt.checkpoint(self.seg1, x, use_reentrant=False)
        x = ckpt.checkpoint(self.seg2, x, use_reentrant=False)
        return x

model.features = CheckpointedFeatures(_seg1, _seg2)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}")
print(f"Gradient checkpointing: 2 segments")

model = model.to(DEVICE)

Loaded: /kaggle/input/datasets/raahad/mobilenetv2-parking-occupancy-phase1-weights/phase1_weights.pth
Trainable: 2,226,434 / 2,226,434
Gradient checkpointing: 2 segments


## Training and Validation Functions

In [5]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, epoch, total_epochs):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_val    = loss.item()
        total_loss += loss_val * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
        del images, labels, outputs, loss
        pbar.set_postfix(loss=f"{loss_val:.4f}", acc=f"{correct/total:.4f}")
    del pbar
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch, total_epochs):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} val  ", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        loss_val    = loss.item()
        total_loss += loss_val * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
        del images, labels, outputs, loss
        pbar.set_postfix(loss=f"{loss_val:.4f}", acc=f"{correct/total:.4f}")
    del pbar
    return total_loss / total, correct / total

## Run Phase 2 Training

In [6]:
os.makedirs(EXPORT_DIR, exist_ok=True)
BEST_WEIGHTS = os.path.join(EXPORT_DIR, WEIGHTS_FILENAME)
CSV_PATH     = os.path.join(EXPORT_DIR, 'training_log.csv')

backbone  = model  # single GPU, no DataParallel wrapper
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler    = GradScaler('cuda', enabled=USE_AMP)

best_val_acc = 0.0
best_epoch   = 0
history      = []

log_mem("before training")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, DEVICE, epoch, EPOCHS)
    vl_loss, vl_acc = evaluate(
        model, val_loader, criterion, DEVICE, epoch, EPOCHS)

    print(f"Epoch {epoch}/{EPOCHS}   "
          f"train loss {tr_loss:.4f}  acc {tr_acc:.4f}   "
          f"val loss {vl_loss:.4f}  acc {vl_acc:.4f}", end="")

    history.append({'epoch': epoch,
                    'train_loss': tr_loss, 'train_acc': tr_acc,
                    'val_loss'  : vl_loss, 'val_acc'  : vl_acc})

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_epoch   = epoch
        torch.save(backbone.state_dict(), BEST_WEIGHTS)
        print("  <- best saved")
    else:
        print()

    gc.collect()
    torch.cuda.empty_cache()
    log_mem(f"epoch {epoch} done")

with open(CSV_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['epoch','train_loss','train_acc','val_loss','val_acc'])
    writer.writeheader()
    writer.writerows(history)

print(f"\nBest: epoch {best_epoch}  val_acc {best_val_acc:.4f}")
print(f"Phase 1 baseline: 0.9728")
print(f"Best weights: {BEST_WEIGHTS}")
print(f"CSV log:      {CSV_PATH}")

[MEM before training] RAM 1.6/31.3 GB (6.6%)
  GPU: allocated 0.01 GB, reserved 0.03 GB


/tmp/ipykernel_58/2329432904.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=USE_AMP)
Epoch 1/10 train:   0%|          | 0/4880 [00:00<?, ?it/s]/tmp/ipykernel_58/1144718939.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
Epoch 1/10 val  :   0%|          | 0/1441 [00:00<?, ?it/s]                                     /tmp/ipykernel_58/1144718939.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch 1/10   train loss 0.0134  acc 0.9966   val loss 0.0560  acc 0.9816  <- best saved
[MEM epoch 1 done] RAM 3.8/31.3 GB (15.5%)
  GPU: allocated 0.05 GB, reserved 0.11 GB


Epoch 2/10   train loss 0.0074  acc 0.9984   val loss 0.0253  acc 0.9915  <- best saved
[MEM epoch 2 done] RAM 4.1/31.3 GB (16.5%)
  GPU: allocated 0.05 GB, reserved 0.12 GB


Epoch 3/10   train loss 0.0056  acc 0.9987   val loss 0.0314  acc 0.9900
[MEM epoch 3 done] RAM 3.7/31.3 GB (15.4%)
  GPU: allocated 0.05 GB, reserved 0.11 GB


...

Epoch 6/10   train loss 0.0034  acc 0.9992   val loss 0.0374  acc 0.9886
[MEM epoch 6 done] RAM 3.7/31.3 GB (15.4%)
  GPU: allocated 0.05 GB, reserved 0.12 GB


Epoch 7/10   train loss 0.0032  acc 0.9992   val loss 0.0868  acc 0.9729
[MEM epoch 7 done] RAM 3.8/31.3 GB (15.6%)
  GPU: allocated 0.05 GB, reserved 0.11 GB


Epoch 8/10   train loss 0.0027  acc 0.9993   val loss 0.0385  acc 0.9897
[MEM epoch 8 done] RAM 3.5/31.3 GB (14.7%)
  GPU: allocated 0.05 GB, reserved 0.12 GB


Epoch 9/10   train loss 0.0024  acc 0.9993   val loss 0.0306  acc 0.9918  <- best saved
[MEM epoch 9 done] RAM 3.7/31.3 GB (15.5%)
  GPU: allocated 0.05 GB, reserved 0.11 GB


Epoch 10/10   train loss 0.0022  acc 0.9994   val loss 0.0378  acc 0.9893
[MEM epoch 10 done] RAM 3.6/31.3 GB (15.0%)
  GPU: allocated 0.05 GB, reserved 0.12 GB

Best: epoch 9  val_acc 0.9918
Phase 1 baseline: 0.9728
Best weights: /kaggle/working/mobilenetv2-parking-occupancy-phase2-weights/phase2_weights.pth
CSV log:      /kaggle/working/mobilenetv2-parking-occupancy-phase2-weights/training_log.csv
